In [ ]:
# ============================================================
# CENTRO DE CONTROL DE MOVILIDAD URBANA — CIUDAD NOVA
# Dashboard interactivo para Google Colab
# ============================================================

# ============================================================
# 1. INSTALACIÓN
# ============================================================
!pip -q install dash dash-bootstrap-components plotly pandas numpy scipy

# ============================================================
# 2. IMPORTACIONES
# ============================================================
import numpy as np
import pandas as pd
from datetime import datetime
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

from dash import Dash, dcc, html, dash_table, Input, Output
import dash_bootstrap_components as dbc

# ============================================================
# 3. CONFIGURACIÓN
# ============================================================
np.random.seed(42)

N_REGISTROS = 100_000
FECHA_INICIO = pd.Timestamp("2025-09-01")
FECHA_FIN = pd.Timestamp("2026-08-31")

ZONAS = [
    "Centro", "Norte", "Sur", "Oriente", "Occidente",
    "Zona Industrial", "Zona Universitaria", "Zona Residencial"
]

TRANSPORTES = [
    "Automóvil", "Motocicleta", "Autobús", "Metro",
    "Bicicleta", "Taxi", "Caminata"
]

DIAS = [
    "Lunes", "Martes", "Miércoles", "Jueves",
    "Viernes", "Sábado", "Domingo"
]

MESES = [
    "Enero", "Febrero", "Marzo", "Abril", "Mayo", "Junio",
    "Julio", "Agosto", "Septiembre", "Octubre", "Noviembre", "Diciembre"
]

COLORES_TRANSPORTE = {
    "Automóvil": "#4C78A8",
    "Motocicleta": "#F58518",
    "Autobús": "#54A24B",
    "Metro": "#E45756",
    "Bicicleta": "#72B7B2",
    "Taxi": "#EECA3B",
    "Caminata": "#B279A2"
}

TEMA = "plotly_dark"

# ============================================================
# 4. GENERACIÓN DE DATOS SIMULADOS
# ============================================================

fechas = pd.date_range(FECHA_INICIO, FECHA_FIN, freq="D")
fechas_muestra = np.random.choice(fechas.values, size=N_REGISTROS)

df = pd.DataFrame({
    "fecha": pd.to_datetime(fechas_muestra),
    "hora": np.random.choice(
        np.arange(24),
        size=N_REGISTROS,
        p=np.array([
            0.018, 0.012, 0.010, 0.010, 0.014, 0.030,
            0.055, 0.080, 0.070, 0.045, 0.040, 0.040,
            0.045, 0.042, 0.040, 0.045, 0.060, 0.085,
            0.095, 0.080, 0.060, 0.045, 0.035, 0.024
        ]) / np.sum([
            0.018, 0.012, 0.010, 0.010, 0.014, 0.030,
            0.055, 0.080, 0.070, 0.045, 0.040, 0.040,
            0.045, 0.042, 0.040, 0.045, 0.060, 0.085,
            0.095, 0.080, 0.060, 0.045, 0.035, 0.024
        ])
    ),
    "zona_origen": np.random.choice(
        ZONAS, size=N_REGISTROS,
        p=[0.18, 0.13, 0.12, 0.11, 0.11, 0.10, 0.10, 0.15]
    ),
    "zona_destino": np.random.choice(
        ZONAS, size=N_REGISTROS,
        p=[0.18, 0.13, 0.12, 0.11, 0.11, 0.10, 0.10, 0.15]
    ),
    "tipo_transporte": np.random.choice(
        TRANSPORTES, size=N_REGISTROS,
        p=[0.27, 0.16, 0.18, 0.13, 0.08, 0.11, 0.07]
    )
})

# Evitar origen = destino.
mismo_origen = df["zona_origen"].eq(df["zona_destino"])
while mismo_origen.any():
    idx = df.index[mismo_origen]
    df.loc[idx, "zona_destino"] = np.random.choice(ZONAS, size=len(idx))
    mismo_origen = df["zona_origen"].eq(df["zona_destino"])

df["dia_semana_num"] = df["fecha"].dt.dayofweek
df["dia_semana"] = df["dia_semana_num"].map(dict(enumerate(DIAS)))
df["mes_num"] = df["fecha"].dt.month
df["mes"] = df["mes_num"].map(dict(enumerate(MESES, start=1)))
df["mes_periodo"] = df["fecha"].dt.to_period("M").astype(str)

# Demanda por hora: picos de entrada y salida laboral.
efecto_hora = {
    0: 0.25, 1: 0.18, 2: 0.16, 3: 0.15, 4: 0.20, 5: 0.35,
    6: 0.75, 7: 1.30, 8: 1.45, 9: 0.90, 10: 0.70, 11: 0.72,
    12: 0.82, 13: 0.78, 14: 0.70, 15: 0.76, 16: 0.95,
    17: 1.25, 18: 1.55, 19: 1.42, 20: 1.05, 21: 0.78,
    22: 0.58, 23: 0.40
}
df["factor_hora"] = df["hora"].map(efecto_hora)

df["es_fin_semana"] = df["dia_semana_num"] >= 5
df["factor_dia"] = np.where(df["es_fin_semana"], 0.72, 1.0)

factor_zona = {
    "Centro": 1.25, "Norte": 1.00, "Sur": 1.05, "Oriente": 0.98,
    "Occidente": 0.95, "Zona Industrial": 1.18,
    "Zona Universitaria": 1.10, "Zona Residencial": 0.90
}

congestion_zona = {
    "Centro": 1.15, "Norte": 0.92, "Sur": 1.02, "Oriente": 0.98,
    "Occidente": 0.94, "Zona Industrial": 1.28,
    "Zona Universitaria": 1.08, "Zona Residencial": 0.82
}

df["factor_zona"] = df["zona_origen"].map(factor_zona)
df["factor_congestion_zona"] = df["zona_origen"].map(congestion_zona)

base_demanda = 2.2 * df["factor_hora"] * df["factor_dia"] * df["factor_zona"]

es_publico = df["tipo_transporte"].isin(["Autobús", "Metro"])
base_demanda += es_publico.astype(float) * df["factor_hora"] * 0.9

dia_anual = (df["fecha"] - FECHA_INICIO).dt.days
estacionalidad = 1 + 0.08 * np.sin(2 * np.pi * dia_anual / 365.25)
base_demanda *= estacionalidad

df["cantidad_viajes"] = np.maximum(
    1, np.random.poisson(np.clip(base_demanda, 0.5, 20))
)

# Distancias realistas según ubicación relativa.
centroides = {
    "Centro": (0.0, 0.0),
    "Norte": (2.8, 3.2),
    "Sur": (-2.5, -3.0),
    "Oriente": (3.4, -0.8),
    "Occidente": (-3.3, 0.7),
    "Zona Industrial": (-4.5, -1.8),
    "Zona Universitaria": (1.0, 2.5),
    "Zona Residencial": (2.1, -2.6)
}

x_origen = df["zona_origen"].map({k: v[0] for k, v in centroides.items()}).to_numpy()
y_origen = df["zona_origen"].map({k: v[1] for k, v in centroides.items()}).to_numpy()
x_destino = df["zona_destino"].map({k: v[0] for k, v in centroides.items()}).to_numpy()
y_destino = df["zona_destino"].map({k: v[1] for k, v in centroides.items()}).to_numpy()

distancia_base = (
    np.sqrt((x_destino - x_origen) ** 2 + (y_destino - y_origen) ** 2)
    * 1.65 + 1.2
)

df["distancia_km"] = np.round(
    np.clip(
        distancia_base * np.random.lognormal(0, 0.16, N_REGISTROS),
        0.8, 25
    ),
    2
)

# Índice de tráfico correlacionado con hora, zona y día.
pico = (
    0.55 * df["factor_hora"]
    + 0.35 * df["factor_congestion_zona"]
    + np.where(df["es_fin_semana"], -0.35, 0.20)
)

pico += np.where(
    (df["zona_origen"] == "Zona Industrial") &
    (df["hora"].between(6, 18)),
    0.55, 0
)

indice = 42 + 19 * pico + np.random.normal(0, 8, N_REGISTROS)

# Anomalías intencionales.
anomalia_1 = (
    (df["fecha"] >= pd.Timestamp("2026-02-10")) &
    (df["fecha"] <= pd.Timestamp("2026-02-12")) &
    (df["hora"].between(17, 20))
)

anomalia_2 = (
    (df["fecha"] >= pd.Timestamp("2026-06-18")) &
    (df["fecha"] <= pd.Timestamp("2026-06-19")) &
    (df["hora"].between(7, 9))
)

indice += np.where(anomalia_1, 24, 0)
indice += np.where(anomalia_2, 18, 0)

df["indice_trafico"] = np.round(np.clip(indice, 5, 100), 2)

df["nivel_congestion"] = pd.cut(
    df["indice_trafico"],
    bins=[-np.inf, 35, 55, 75, np.inf],
    labels=["Baja", "Media", "Alta", "Crítica"]
).astype(str)

# Velocidad según medio de transporte.
velocidad_base = {
    "Automóvil": 46, "Motocicleta": 50, "Autobús": 31,
    "Metro": 58, "Bicicleta": 18, "Taxi": 43, "Caminata": 5.2
}

df["velocidad_base"] = df["tipo_transporte"].map(velocidad_base)

reduccion_congestion = np.where(
    df["tipo_transporte"].isin(
        ["Automóvil", "Motocicleta", "Autobús", "Taxi"]
    ),
    0.48 * df["indice_trafico"],
    np.where(
        df["tipo_transporte"] == "Bicicleta",
        0.08 * df["indice_trafico"],
        0
    )
)

df["velocidad_promedio_kmh"] = (
    df["velocidad_base"]
    - reduccion_congestion
    + np.random.normal(0, 3.5, N_REGISTROS)
)

df["velocidad_promedio_kmh"] = np.round(
    np.clip(df["velocidad_promedio_kmh"], 3, 65), 2
)

# Duración relacionada con distancia, velocidad y congestión.
tiempo_base = (
    df["distancia_km"] / df["velocidad_promedio_kmh"]
) * 60

tiempo_extra = np.where(
    df["tipo_transporte"].isin(
        ["Automóvil", "Motocicleta", "Autobús", "Taxi"]
    ),
    df["indice_trafico"] * 0.08,
    df["indice_trafico"] * 0.015
)

df["duracion_minutos"] = np.round(
    np.clip(
        tiempo_base + tiempo_extra + np.random.normal(1.5, 2.0, N_REGISTROS),
        3, 120
    ),
    2
)

# Accidentes e incidentes.
riesgo = (
    0.00035 * df["cantidad_viajes"] *
    (1 + df["indice_trafico"] / 65)
)

riesgo *= np.where(
    df["tipo_transporte"] == "Motocicleta", 1.65, 1.0
)
riesgo *= np.where(
    df["tipo_transporte"].isin(["Automóvil", "Taxi"]), 1.20, 1.0
)

riesgo = np.clip(riesgo, 0, 0.12)
df["accidentes"] = np.random.binomial(1, riesgo)

riesgo_incidente = (
    0.012
    + 0.006 * (df["indice_trafico"] / 100)
    + 0.002 * df["cantidad_viajes"]
)

riesgo_incidente *= np.where(
    df["zona_origen"] == "Zona Industrial", 1.30, 1.0
)
riesgo_incidente = np.clip(riesgo_incidente, 0, 0.20)

df["incidentes"] = np.random.poisson(riesgo_incidente)

df.loc[anomalia_1, "incidentes"] += np.random.binomial(
    2, 0.35, anomalia_1.sum()
)
df.loc[anomalia_2, "incidentes"] += np.random.binomial(
    2, 0.25, anomalia_2.sum()
)

# Pasajeros estimados.
ocupacion = {
    "Automóvil": 1.55, "Motocicleta": 1.10, "Autobús": 32,
    "Metro": 120, "Bicicleta": 1, "Taxi": 1.65, "Caminata": 1
}

df["pasajeros"] = np.maximum(
    1,
    np.round(
        df["cantidad_viajes"]
        * df["tipo_transporte"].map(ocupacion)
        * np.random.normal(1, 0.12, N_REGISTROS)
    ).astype(int)
)

# Emisiones estimadas.
emision_por_km = {
    "Automóvil": 0.192,
    "Motocicleta": 0.103,
    "Autobús": 0.085,
    "Metro": 0.035,
    "Bicicleta": 0.0,
    "Taxi": 0.210,
    "Caminata": 0.0
}

df["emisiones_co2_kg"] = (
    df["distancia_km"]
    * df["cantidad_viajes"]
    * df["tipo_transporte"].map(emision_por_km)
    * np.random.normal(1, 0.08, N_REGISTROS)
)

df["emisiones_co2_kg"] = np.round(
    np.clip(df["emisiones_co2_kg"], 0, None), 3
)

# Costo estimado.
tarifa_base = {
    "Automóvil": 4200, "Motocicleta": 2800, "Autobús": 3200,
    "Metro": 3500, "Bicicleta": 900, "Taxi": 6500, "Caminata": 0
}

df["costo_promedio"] = (
    df["tipo_transporte"].map(tarifa_base)
    + df["distancia_km"] * np.where(
        df["tipo_transporte"].isin(
            ["Automóvil", "Taxi", "Motocicleta"]
        ),
        500, 180
    )
    + np.random.normal(0, 500, N_REGISTROS)
)

df["costo_promedio"] = np.round(
    np.clip(df["costo_promedio"], 0, None), 0
)

# Satisfacción inversamente relacionada con congestión y duración.
satisfaccion = (
    86
    - 0.24 * df["indice_trafico"]
    - 0.10 * df["duracion_minutos"]
    + np.where(
        df["tipo_transporte"].isin(
            ["Metro", "Bicicleta", "Caminata"]
        ), 5, 0
    )
    + np.random.normal(0, 5, N_REGISTROS)
)

df["satisfaccion_usuario"] = np.round(
    np.clip(satisfaccion, 25, 100), 1
)

# Variables auxiliares.
df["fecha_hora"] = df["fecha"] + pd.to_timedelta(df["hora"], unit="h")
df["hora_texto"] = df["hora"].astype(str).str.zfill(2) + ":00"
df["periodo"] = df["fecha"].dt.strftime("%Y-%m")

df = df.sort_values("fecha_hora").reset_index(drop=True)

print(f"Dataset creado correctamente: {len(df):,} registros.")
print(f"Período: {df['fecha'].min():%d/%m/%Y} — {df['fecha'].max():%d/%m/%Y}")
print(f"Columnas: {len(df.columns)}")

# ============================================================
# 5. FUNCIONES AUXILIARES
# ============================================================

def filtrar_datos(
    fecha_inicio, fecha_fin, zonas, transportes,
    dias, meses, niveles
):
    datos = df[
        (df["fecha"] >= pd.to_datetime(fecha_inicio)) &
        (df["fecha"] <= pd.to_datetime(fecha_fin))
    ]

    if zonas:
        datos = datos[datos["zona_origen"].isin(zonas)]

    if transportes:
        datos = datos[datos["tipo_transporte"].isin(transportes)]

    if dias:
        datos = datos[datos["dia_semana"].isin(dias)]

    if meses:
        datos = datos[datos["mes"].isin(meses)]

    if niveles:
        datos = datos[datos["nivel_congestion"].isin(niveles)]

    return datos


def tarjeta_kpi(titulo, valor, subtitulo=""):
    return html.Div(
        [
            html.Div(titulo, className="kpi-title"),
            html.Div(valor, className="kpi-value"),
            html.Div(subtitulo, className="kpi-subtitle"),
        ],
        className="kpi-card"
    )


def estilo_figura(fig, titulo="", altura=360):
    fig.update_layout(
        template="plotly_dark",
        height=altura,
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
        font=dict(family="Arial", color="#E8EEF5"),
        title=dict(
            text=titulo,
            x=0.02,
            xanchor="left",
            font=dict(size=17)
        ),
        margin=dict(l=50, r=35, t=55, b=45),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.01,
            xanchor="left",
            x=0
        ),
        hoverlabel=dict(
            bgcolor="#17212B",
            font_size=12
        )
    )
    return fig


def texto_numero(valor, decimales=0):
    return (
        f"{valor:,.{decimales}f}"
        .replace(",", "X")
        .replace(".", ",")
        .replace("X", ".")
    )


def generar_insights(datos):
    if datos.empty:
        return [
            html.Div(
                "No hay datos para los filtros seleccionados.",
                className="insight-empty"
            )
        ]

    congestion_zona = (
        datos.groupby("zona_origen")["indice_trafico"]
        .mean()
        .sort_values(ascending=False)
    )
    zona = congestion_zona.index[0]
    zona_valor = congestion_zona.iloc[0]

    viajes_hora = datos.groupby("hora")["cantidad_viajes"].sum()
    hora = viajes_hora.idxmax()
    hora_viajes = viajes_hora.max()

    viajes_transporte = (
        datos.groupby("tipo_transporte")["cantidad_viajes"]
        .sum()
        .sort_values(ascending=False)
    )
    transporte = viajes_transporte.index[0]

    viajes_dia = (
        datos.groupby("dia_semana")["cantidad_viajes"]
        .sum()
        .reindex(DIAS)
    )
    dia = viajes_dia.idxmax()

    emisiones_transporte = (
        datos.groupby("tipo_transporte")["emisiones_co2_kg"]
        .sum()
        .sort_values(ascending=False)
    )
    mayor_emisor = emisiones_transporte.index[0]
    mayor_emisor_valor = emisiones_transporte.iloc[0]

    evolucion = (
        datos.groupby("fecha")["emisiones_co2_kg"]
        .sum()
        .sort_index()
    )

    if len(evolucion) >= 14:
        mitad = len(evolucion) // 2
        primera = evolucion.iloc[:mitad].mean()
        segunda = evolucion.iloc[mitad:].mean()
        cambio = ((segunda - primera) / primera * 100) if primera else 0
    else:
        cambio = 0

    if cambio > 2:
        tendencia = f"aumentaron {cambio:.1f}%"
    elif cambio < -2:
        tendencia = f"disminuyeron {abs(cambio):.1f}%"
    else:
        tendencia = "se mantuvieron relativamente estables"

    datos_insight = [
        ("01", "Mayor congestión",
         f"{zona} registra el índice promedio más alto: {zona_valor:.1f}/100."),
        ("02", "Hora de mayor demanda",
         f"{hora:02d}:00 concentra aproximadamente {texto_numero(hora_viajes)} viajes."),
        ("03", "Transporte dominante",
         f"{transporte} es el medio con mayor cantidad de viajes."),
        ("04", "Día con mayor demanda",
         f"{dia} presenta la mayor demanda acumulada."),
        ("05", "Principal fuente de CO₂",
         f"{mayor_emisor} aporta aproximadamente {mayor_emisor_valor:,.1f} kg de CO₂."),
        ("06", "Tendencia de emisiones",
         f"En la comparación temporal, las emisiones {tendencia}.")
    ]

    return [
        html.Div(
            [
                html.Div(numero, className="insight-number"),
                html.Div(
                    [
                        html.Div(titulo, className="insight-title"),
                        html.Div(texto, className="insight-text")
                    ]
                )
            ],
            className="insight-item"
        )
        for numero, titulo, texto in datos_insight
    ]

# ============================================================
# 6. APLICACIÓN DASH
# ============================================================

app = Dash(
    __name__,
    external_stylesheets=[dbc.themes.DARKLY],
    suppress_callback_exceptions=True
)

app.title = "Ciudad Nova | Centro de Control de Movilidad"

# ============================================================
# 7. LAYOUT
# ============================================================

app.layout = html.Div(
    [
        html.Div(
            [
                html.Div(
                    [
                        html.Div(
                            "NOVA MOBILITY INTELLIGENCE",
                            className="eyebrow"
                        ),
                        html.H1(
                            "CENTRO DE CONTROL DE MOVILIDAD URBANA"
                        ),
                        html.P(
                            "Panel ejecutivo de movilidad inteligente — Ciudad Nova",
                            className="subtitle"
                        )
                    ],
                    className="header-text"
                ),
                html.Div(
                    [
                        html.Div(
                            "PERÍODO ANALIZADO",
                            className="period-label"
                        ),
                        html.Div(
                            f"{FECHA_INICIO:%d/%m/%Y} — {FECHA_FIN:%d/%m/%Y}",
                            className="period-value"
                        ),
                        html.Div(
                            f"{N_REGISTROS:,} registros simulados",
                            className="period-records"
                        )
                    ],
                    className="header-period"
                )
            ],
            className="dashboard-header"
        ),

        html.Div("FILTROS DE ANÁLISIS", className="section-label"),

        html.Div(
            [
                html.Div(
                    [
                        html.Label("Rango de fechas"),
                        dcc.DatePickerRange(
                            id="filtro-fechas",
                            min_date_allowed=FECHA_INICIO,
                            max_date_allowed=FECHA_FIN,
                            start_date=FECHA_INICIO,
                            end_date=FECHA_FIN,
                            display_format="DD/MM/YYYY"
                        )
                    ],
                    className="filter-item filter-date"
                ),

                html.Div(
                    [
                        html.Label("Zona"),
                        dcc.Dropdown(
                            id="filtro-zona",
                            options=[
                                {"label": z, "value": z} for z in ZONAS
                            ],
                            value=ZONAS,
                            multi=True
                        )
                    ],
                    className="filter-item"
                ),

                html.Div(
                    [
                        html.Label("Tipo de transporte"),
                        dcc.Dropdown(
                            id="filtro-transporte",
                            options=[
                                {"label": t, "value": t}
                                for t in TRANSPORTES
                            ],
                            value=TRANSPORTES,
                            multi=True
                        )
                    ],
                    className="filter-item"
                ),

                html.Div(
                    [
                        html.Label("Día de la semana"),
                        dcc.Dropdown(
                            id="filtro-dia",
                            options=[
                                {"label": d, "value": d} for d in DIAS
                            ],
                            value=DIAS,
                            multi=True
                        )
                    ],
                    className="filter-item"
                ),

                html.Div(
                    [
                        html.Label("Mes"),
                        dcc.Dropdown(
                            id="filtro-mes",
                            options=[
                                {"label": m, "value": m} for m in MESES
                            ],
                            value=MESES,
                            multi=True
                        )
                    ],
                    className="filter-item"
                ),

                html.Div(
                    [
                        html.Label("Nivel de congestión"),
                        dcc.Dropdown(
                            id="filtro-congestion",
                            options=[
                                {"label": n, "value": n}
                                for n in ["Baja", "Media", "Alta", "Crítica"]
                            ],
                            value=["Baja", "Media", "Alta", "Crítica"],
                            multi=True
                        )
                    ],
                    className="filter-item"
                )
            ],
            className="filters-grid panel filters-panel"
        ),

        html.Div(
            "INDICADORES CLAVE DE DESEMPEÑO",
            className="section-label"
        ),
        html.Div(id="contenedor-kpis", className="kpi-grid"),

        html.Div("EVOLUCIÓN TEMPORAL", className="section-label"),
        html.Div(
            dcc.Graph(
                id="grafico-evolucion",
                config={
                    "displaylogo": False,
                    "modeBarButtonsToAdd": ["select2d", "lasso2d"]
                }
            ),
            className="panel graph-panel"
        ),

        html.Div(
            "MAPA DE CALOR DE CONGESTIÓN",
            className="section-label"
        ),
        html.Div(
            [
                html.P(
                    "Índice promedio de tráfico por día de la semana y hora del día.",
                    className="chart-description"
                ),
                dcc.Graph(
                    id="grafico-heatmap",
                    config={"displaylogo": False}
                )
            ],
            className="panel graph-panel"
        ),

        html.Div("ANÁLISIS POR ZONA", className="section-label"),
        html.Div(
            [
                html.Div(
                    dcc.Graph(
                        id="grafico-zonas",
                        config={"displaylogo": False}
                    ),
                    className="half-panel"
                ),
                html.Div(
                    dcc.Graph(
                        id="grafico-zonas-treemap",
                        config={"displaylogo": False}
                    ),
                    className="half-panel"
                )
            ],
            className="two-columns"
        ),

        html.Div(
            "ANÁLISIS DE TRANSPORTE",
            className="section-label"
        ),
        html.Div(
            [
                html.Div(
                    dcc.Graph(
                        id="grafico-donut",
                        config={"displaylogo": False}
                    ),
                    className="half-panel"
                ),
                html.Div(
                    dcc.Graph(
                        id="grafico-transporte-barras",
                        config={"displaylogo": False}
                    ),
                    className="half-panel"
                )
            ],
            className="two-columns"
        ),

        html.Div(
            dcc.Graph(
                id="grafico-boxplot",
                config={"displaylogo": False}
            ),
            className="panel graph-panel"
        ),

        html.Div(
            "ANÁLISIS DE EMISIONES DE CO₂",
            className="section-label"
        ),
        html.Div(
            [
                html.Div(
                    dcc.Graph(
                        id="grafico-emisiones-transporte",
                        config={"displaylogo": False}
                    ),
                    className="half-panel"
                ),
                html.Div(
                    dcc.Graph(
                        id="grafico-emisiones-zona",
                        config={"displaylogo": False}
                    ),
                    className="half-panel"
                )
            ],
            className="two-columns"
        ),

        html.Div(
            dcc.Graph(
                id="grafico-emisiones-tiempo",
                config={"displaylogo": False}
            ),
            className="panel graph-panel"
        ),

        html.Div(
            "SEGURIDAD E INCIDENTES",
            className="section-label"
        ),
        html.Div(
            [
                html.Div(
                    dcc.Graph(
                        id="grafico-incidentes-tiempo",
                        config={"displaylogo": False}
                    ),
                    className="half-panel"
                ),
                html.Div(
                    dcc.Graph(
                        id="grafico-incidentes-hora",
                        config={"displaylogo": False}
                    ),
                    className="half-panel"
                )
            ],
            className="two-columns"
        ),

        html.Div("INSIGHTS PRINCIPALES", className="section-label"),
        html.Div(
            id="contenedor-insights",
            className="insights-grid"
        ),

        html.Div("DETALLE DE REGISTROS", className="section-label"),
        html.Div(
            [
                html.P(
                    "La tabla responde a todos los filtros seleccionados. "
                    "Permite búsqueda, ordenamiento, selección y paginación.",
                    className="chart-description"
                ),
                dash_table.DataTable(
                    id="tabla-detalle",
                    columns=[
                        {"name": "Fecha", "id": "fecha"},
                        {"name": "Hora", "id": "hora_texto"},
                        {"name": "Día", "id": "dia_semana"},
                        {"name": "Origen", "id": "zona_origen"},
                        {"name": "Destino", "id": "zona_destino"},
                        {"name": "Transporte", "id": "tipo_transporte"},
                        {"name": "Viajes", "id": "cantidad_viajes"},
                        {"name": "Velocidad km/h", "id": "velocidad_promedio_kmh"},
                        {"name": "Distancia km", "id": "distancia_km"},
                        {"name": "Duración min", "id": "duracion_minutos"},
                        {"name": "Congestión", "id": "nivel_congestion"},
                        {"name": "Índice tráfico", "id": "indice_trafico"},
                        {"name": "Accidentes", "id": "accidentes"},
                        {"name": "Incidentes", "id": "incidentes"},
                        {"name": "CO₂ kg", "id": "emisiones_co2_kg"},
                        {"name": "Pasajeros", "id": "pasajeros"},
                        {"name": "Costo", "id": "costo_promedio"},
                        {"name": "Satisfacción", "id": "satisfaccion_usuario"}
                    ],
                    data=[],
                    page_size=15,
                    page_action="native",
                    sort_action="native",
                    filter_action="native",
                    row_selectable="single",
                    selected_rows=[],
                    style_table={
                        "overflowX": "auto",
                        "maxHeight": "600px",
                        "overflowY": "auto"
                    },
                    style_header={
                        "backgroundColor": "#182430",
                        "color": "#FFFFFF",
                        "fontWeight": "bold",
                        "border": "1px solid #2C3E50",
                        "textAlign": "center"
                    },
                    style_cell={
                        "backgroundColor": "#101820",
                        "color": "#DDE6EF",
                        "border": "1px solid #263442",
                        "fontSize": "12px",
                        "padding": "9px",
                        "textAlign": "left",
                        "minWidth": "100px"
                    },
                    style_data_conditional=[
                        {
                            "if": {
                                "filter_query":
                                '{nivel_congestion} = "Crítica"'
                            },
                            "backgroundColor": "rgba(231, 76, 60, 0.20)"
                        },
                        {
                            "if": {
                                "filter_query":
                                '{nivel_congestion} = "Alta"'
                            },
                            "backgroundColor": "rgba(230, 126, 34, 0.16)"
                        }
                    ]
                )
            ],
            className="panel table-panel"
        ),

        html.Div(
            "Ciudad Nova • Centro de Control de Movilidad • "
            "Datos simulados para fines analíticos",
            className="footer"
        )
    ],
    className="dashboard"
)

# ============================================================
# 8. CALLBACKS
# ============================================================

@app.callback(
    [
        Output("contenedor-kpis", "children"),
        Output("grafico-evolucion", "figure"),
        Output("grafico-heatmap", "figure"),
        Output("grafico-zonas", "figure"),
        Output("grafico-zonas-treemap", "figure"),
        Output("grafico-donut", "figure"),
        Output("grafico-transporte-barras", "figure"),
        Output("grafico-boxplot", "figure"),
        Output("grafico-emisiones-transporte", "figure"),
        Output("grafico-emisiones-zona", "figure"),
        Output("grafico-emisiones-tiempo", "figure"),
        Output("grafico-incidentes-tiempo", "figure"),
        Output("grafico-incidentes-hora", "figure"),
        Output("contenedor-insights", "children"),
        Output("tabla-detalle", "data")
    ],
    [
        Input("filtro-fechas", "start_date"),
        Input("filtro-fechas", "end_date"),
        Input("filtro-zona", "value"),
        Input("filtro-transporte", "value"),
        Input("filtro-dia", "value"),
        Input("filtro-mes", "value"),
        Input("filtro-congestion", "value")
    ]
)
def actualizar_dashboard(
    fecha_inicio, fecha_fin, zonas, transportes,
    dias, meses, niveles
):
    datos = filtrar_datos(
        fecha_inicio, fecha_fin, zonas, transportes,
        dias, meses, niveles
    )

    if datos.empty:
        vacio = go.Figure()
        estilo_figura(
            vacio,
            "Sin datos para los filtros seleccionados"
        )
        vacio.add_annotation(
            text="No existen registros que cumplan los filtros.",
            x=0.5, y=0.5,
            xref="paper", yref="paper",
            showarrow=False,
            font=dict(size=16)
        )

        kpis = [
            tarjeta_kpi("TOTAL DE VIAJES", "0", "Sin registros"),
            tarjeta_kpi("VELOCIDAD PROMEDIO", "— km/h", "Sin registros"),
            tarjeta_kpi("TIEMPO PROMEDIO", "— min", "Sin registros"),
            tarjeta_kpi("ÍNDICE DE CONGESTIÓN", "— /100", "Sin registros"),
            tarjeta_kpi("EMISIONES CO₂", "— kg", "Sin registros"),
            tarjeta_kpi("INCIDENTES", "0", "Sin registros"),
            tarjeta_kpi("SATISFACCIÓN", "— /100", "Sin registros")
        ]

        return (
            kpis, vacio, vacio, vacio, vacio, vacio, vacio, vacio,
            vacio, vacio, vacio, vacio, vacio,
            [html.Div(
                "No hay insights disponibles.",
                className="insight-empty"
            )],
            []
        )

    # --------------------------------------------------------
    # KPI
    # --------------------------------------------------------
    total_viajes = datos["cantidad_viajes"].sum()
    velocidad = datos["velocidad_promedio_kmh"].mean()
    duracion = datos["duracion_minutos"].mean()
    congestion = datos["indice_trafico"].mean()
    emisiones = datos["emisiones_co2_kg"].sum()
    incidentes = datos["incidentes"].sum()
    satisfaccion = datos["satisfaccion_usuario"].mean()

    kpis = [
        tarjeta_kpi(
            "TOTAL DE VIAJES",
            texto_numero(total_viajes),
            f"{len(datos):,} registros filtrados"
        ),
        tarjeta_kpi(
            "VELOCIDAD PROMEDIO",
            f"{velocidad:.1f} km/h",
            "Meta referencial > 35 km/h"
        ),
        tarjeta_kpi(
            "TIEMPO PROMEDIO",
            f"{duracion:.1f} min",
            "Por desplazamiento"
        ),
        tarjeta_kpi(
            "ÍNDICE DE CONGESTIÓN",
            f"{congestion:.1f} /100",
            "Menor es mejor"
        ),
        tarjeta_kpi(
            "EMISIONES CO₂",
            f"{texto_numero(emisiones, 1)} kg",
            "Estimación agregada"
        ),
        tarjeta_kpi(
            "INCIDENTES",
            texto_numero(incidentes),
            f"{datos['accidentes'].sum():,.0f} accidentes"
        ),
        tarjeta_kpi(
            "SATISFACCIÓN",
            f"{satisfaccion:.1f} /100",
            "Percepción estimada"
        )
    ]

    # --------------------------------------------------------
    # EVOLUCIÓN TEMPORAL
    # --------------------------------------------------------
    diario = datos.groupby("fecha").agg(
        viajes=("cantidad_viajes", "sum"),
        velocidad=("velocidad_promedio_kmh", "mean"),
        congestion=("indice_trafico", "mean"),
        emisiones=("emisiones_co2_kg", "sum")
    ).reset_index()

    fig_evolucion = make_subplots(
        specs=[[{"secondary_y": True}]]
    )

    fig_evolucion.add_trace(
        go.Scatter(
            x=diario["fecha"],
            y=diario["viajes"],
            name="Viajes",
            mode="lines",
            line=dict(width=2.5),
            hovertemplate=(
                "%{x|%d/%m/%Y}<br>"
                "Viajes: %{y:,.0f}<extra></extra>"
            )
        ),
        secondary_y=False
    )

    fig_evolucion.add_trace(
        go.Scatter(
            x=diario["fecha"],
            y=diario["emisiones"],
            name="CO₂ (kg)",
            mode="lines",
            line=dict(width=2),
            hovertemplate=(
                "%{x|%d/%m/%Y}<br>"
                "CO₂: %{y:,.1f} kg<extra></extra>"
            )
        ),
        secondary_y=False
    )

    fig_evolucion.add_trace(
        go.Scatter(
            x=diario["fecha"],
            y=diario["velocidad"],
            name="Velocidad",
            mode="lines",
            line=dict(width=2, dash="dot"),
            hovertemplate=(
                "%{x|%d/%m/%Y}<br>"
                "Velocidad: %{y:.1f} km/h<extra></extra>"
            )
        ),
        secondary_y=True
    )

    fig_evolucion.add_trace(
        go.Scatter(
            x=diario["fecha"],
            y=diario["congestion"],
            name="Congestión",
            mode="lines",
            line=dict(width=2, dash="dash"),
            hovertemplate=(
                "%{x|%d/%m/%Y}<br>"
                "Congestión: %{y:.1f}/100<extra></extra>"
            )
        ),
        secondary_y=True
    )

    fig_evolucion.update_yaxes(
        title_text="Viajes / emisiones",
        secondary_y=False
    )
    fig_evolucion.update_yaxes(
        title_text="Velocidad / congestión",
        secondary_y=True
    )
    estilo_figura(
        fig_evolucion,
        "Evolución diaria de la movilidad",
        430
    )

    # --------------------------------------------------------
    # HEATMAP
    # --------------------------------------------------------
    heat = datos.pivot_table(
        index="dia_semana",
        columns="hora",
        values="indice_trafico",
        aggfunc="mean"
    ).reindex(DIAS)

    fig_heatmap = go.Figure(
        go.Heatmap(
            z=heat.values,
            x=[f"{h:02d}:00" for h in heat.columns],
            y=heat.index,
            colorscale=[
                [0.0, "#2ECC71"],
                [0.35, "#F1C40F"],
                [0.65, "#E67E22"],
                [1.0, "#E74C3C"]
            ],
            zmin=0,
            zmax=100,
            colorbar=dict(title="Índice"),
            hovertemplate=(
                "Día: %{y}<br>"
                "Hora: %{x}<br>"
                "Índice: %{z:.1f}<extra></extra>"
            )
        )
    )
    estilo_figura(
        fig_heatmap,
        "Mapa de calor: congestión por día y hora",
        400
    )

    # --------------------------------------------------------
    # ZONAS
    # --------------------------------------------------------
    zona_stats = datos.groupby("zona_origen").agg(
        viajes=("cantidad_viajes", "sum"),
        velocidad=("velocidad_promedio_kmh", "mean"),
        congestion=("indice_trafico", "mean"),
        incidentes=("incidentes", "sum"),
        emisiones=("emisiones_co2_kg", "sum")
    ).reset_index().sort_values("viajes", ascending=True)

    fig_zonas = go.Figure(
        go.Bar(
            y=zona_stats["zona_origen"],
            x=zona_stats["viajes"],
            orientation="h",
            name="Viajes",
            hovertemplate=(
                "%{y}<br>Viajes: %{x:,.0f}<extra></extra>"
            )
        )
    )
    fig_zonas.update_xaxes(title="Cantidad de viajes")
    estilo_figura(
        fig_zonas,
        "Demanda de movilidad por zona",
        390
    )

    fig_zonas_treemap = px.treemap(
        zona_stats,
        path=["zona_origen"],
        values="viajes",
        color="congestion",
        color_continuous_scale=[
            "#2ECC71", "#F1C40F", "#E67E22", "#E74C3C"
        ],
        hover_data={
            "viajes": ":,.0f",
            "velocidad": ":.1f",
            "congestion": ":.1f",
            "incidentes": ":,.0f",
            "emisiones": ":,.1f"
        }
    )

    fig_zonas_treemap.update_traces(
        hovertemplate=(
            "<b>%{label}</b><br>"
            "Viajes: %{value:,.0f}<br>"
            "Congestión: %{color:.1f}<extra></extra>"
        )
    )

    estilo_figura(
        fig_zonas_treemap,
        "Mapa jerárquico: demanda y congestión",
        390
    )

    # --------------------------------------------------------
    # TRANSPORTE
    # --------------------------------------------------------
    transporte_stats = datos.groupby("tipo_transporte").agg(
        viajes=("cantidad_viajes", "sum"),
        emisiones=("emisiones_co2_kg", "sum"),
        velocidad=("velocidad_promedio_kmh", "mean"),
        duracion=("duracion_minutos", "mean")
    ).reset_index()

    fig_donut = go.Figure(
        go.Pie(
            labels=transporte_stats["tipo_transporte"],
            values=transporte_stats["viajes"],
            hole=0.62,
            marker=dict(
                colors=[
                    {
                        "Automóvil": "#4C78A8",
                        "Motocicleta": "#F58518",
                        "Autobús": "#54A24B",
                        "Metro": "#E45756",
                        "Bicicleta": "#72B7B2",
                        "Taxi": "#EECA3B",
                        "Caminata": "#B279A2"
                    }[t]
                    for t in transporte_stats["tipo_transporte"]
                ]
            ),
            textinfo="percent",
            hovertemplate=(
                "<b>%{label}</b><br>"
                "Viajes: %{value:,.0f}<br>"
                "Participación: %{percent}<extra></extra>"
            )
        )
    )
    estilo_figura(
        fig_donut,
        "Participación por tipo de transporte",
        390
    )

    fig_transporte = go.Figure()

    fig_transporte.add_trace(
        go.Bar(
            x=transporte_stats["tipo_transporte"],
            y=transporte_stats["velocidad"],
            name="Velocidad",
            hovertemplate=(
                "%{x}<br>%{y:.1f} km/h<extra></extra>"
            )
        )
    )

    fig_transporte.add_trace(
        go.Bar(
            x=transporte_stats["tipo_transporte"],
            y=transporte_stats["duracion"],
            name="Duración",
            hovertemplate=(
                "%{x}<br>%{y:.1f} min<extra></extra>"
            )
        )
    )

    fig_transporte.update_layout(barmode="group")
    fig_transporte.update_yaxes(title="Valor promedio")
    estilo_figura(
        fig_transporte,
        "Velocidad y duración por transporte",
        390
    )

    # Boxplot.
    fig_box = go.Figure()

    for t in TRANSPORTES:
        subset = datos.loc[
            datos["tipo_transporte"] == t,
            "duracion_minutos"
        ]

        if len(subset) > 0:
            fig_box.add_trace(
                go.Box(
                    y=subset,
                    name=t,
                    boxmean=True,
                    hovertemplate=(
                        f"{t}<br>"
                        "Duración: %{y:.1f} min<extra></extra>"
                    )
                )
            )

    fig_box.update_yaxes(title="Duración del viaje (minutos)")
    estilo_figura(
        fig_box,
        "Distribución de duración por medio de transporte",
        430
    )

    # --------------------------------------------------------
    # EMISIONES
    # --------------------------------------------------------
    emisiones_ordenadas = transporte_stats.sort_values(
        "emisiones", ascending=True
    )

    fig_em_trans = go.Figure(
        go.Bar(
            y=emisiones_ordenadas["tipo_transporte"],
            x=emisiones_ordenadas["emisiones"],
            orientation="h",
            hovertemplate=(
                "%{y}<br>"
                "CO₂: %{x:,.1f} kg<extra></extra>"
            )
        )
    )

    fig_em_trans.update_xaxes(
        title="Emisiones estimadas de CO₂ (kg)"
    )
    estilo_figura(
        fig_em_trans,
        "Emisiones por tipo de transporte",
        390
    )

    emisiones_zona = (
        datos.groupby("zona_origen")["emisiones_co2_kg"]
        .sum()
        .reset_index()
        .sort_values("emisiones_co2_kg", ascending=True)
    )

    fig_em_zona = go.Figure(
        go.Bar(
            x=emisiones_zona["emisiones_co2_kg"],
            y=emisiones_zona["zona_origen"],
            orientation="h",
            hovertemplate=(
                "%{y}<br>"
                "CO₂: %{x:,.1f} kg<extra></extra>"
            )
        )
    )

    fig_em_zona.update_xaxes(
        title="Emisiones estimadas (kg)"
    )
    estilo_figura(
        fig_em_zona,
        "Emisiones acumuladas por zona",
        390
    )

    emisiones_tiempo = (
        datos.groupby("fecha")["emisiones_co2_kg"]
        .sum()
        .reset_index()
    )

    fig_em_tiempo = go.Figure(
        go.Scatter(
            x=emisiones_tiempo["fecha"],
            y=emisiones_tiempo["emisiones_co2_kg"],
            mode="lines",
            fill="tozeroy",
            line=dict(width=2.5),
            hovertemplate=(
                "%{x|%d/%m/%Y}<br>"
                "CO₂: %{y:,.1f} kg<extra></extra>"
            )
        )
    )

    fig_em_tiempo.update_yaxes(title="CO₂ (kg)")
    estilo_figura(
        fig_em_tiempo,
        "Evolución diaria de emisiones",
        390
    )

    # --------------------------------------------------------
    # INCIDENTES
    # --------------------------------------------------------
    incidentes_tiempo = datos.groupby("fecha").agg(
        incidentes=("incidentes", "sum"),
        accidentes=("accidentes", "sum")
    ).reset_index()

    fig_inc_tiempo = go.Figure()

    fig_inc_tiempo.add_trace(
        go.Scatter(
            x=incidentes_tiempo["fecha"],
            y=incidentes_tiempo["incidentes"],
            mode="lines",
            name="Incidentes",
            line=dict(width=2.5),
            hovertemplate=(
                "%{x|%d/%m/%Y}<br>"
                "Incidentes: %{y:,.0f}<extra></extra>"
            )
        )
    )

    fig_inc_tiempo.add_trace(
        go.Scatter(
            x=incidentes_tiempo["fecha"],
            y=incidentes_tiempo["accidentes"],
            mode="lines",
            name="Accidentes",
            line=dict(width=2),
            hovertemplate=(
                "%{x|%d/%m/%Y}<br>"
                "Accidentes: %{y:,.0f}<extra></extra>"
            )
        )
    )

    fig_inc_tiempo.update_yaxes(title="Eventos")
    estilo_figura(
        fig_inc_tiempo,
        "Evolución temporal de incidentes y accidentes",
        390
    )

    incidentes_hora = datos.groupby("hora").agg(
        incidentes=("incidentes", "sum"),
        accidentes=("accidentes", "sum")
    ).reset_index()

    fig_inc_hora = go.Figure()

    fig_inc_hora.add_trace(
        go.Bar(
            x=incidentes_hora["hora"],
            y=incidentes_hora["incidentes"],
            name="Incidentes",
            hovertemplate=(
                "Hora: %{x:02d}:00<br>"
                "Incidentes: %{y:,.0f}<extra></extra>"
            )
        )
    )

    fig_inc_hora.add_trace(
        go.Scatter(
            x=incidentes_hora["hora"],
            y=incidentes_hora["accidentes"],
            name="Accidentes",
            mode="lines+markers",
            yaxis="y2",
            hovertemplate=(
                "Hora: %{x:02d}:00<br>"
                "Accidentes: %{y:,.0f}<extra></extra>"
            )
        )
    )

    fig_inc_hora.update_layout(
        yaxis=dict(title="Incidentes"),
        yaxis2=dict(
            title="Accidentes",
            overlaying="y",
            side="right"
        )
    )

    estilo_figura(
        fig_inc_hora,
        "Eventos de seguridad por hora",
        390
    )

    # --------------------------------------------------------
    # INSIGHTS Y TABLA
    # --------------------------------------------------------
    insights = generar_insights(datos)

    tabla = (
        datos.sort_values("fecha_hora", ascending=False)
        .head(5000)
        .copy()
    )

    tabla["fecha"] = tabla["fecha"].dt.strftime("%d/%m/%Y")

    tabla = tabla[
        [
            "fecha", "hora_texto", "dia_semana",
            "zona_origen", "zona_destino", "tipo_transporte",
            "cantidad_viajes", "velocidad_promedio_kmh",
            "distancia_km", "duracion_minutos",
            "nivel_congestion", "indice_trafico",
            "accidentes", "incidentes", "emisiones_co2_kg",
            "pasajeros", "costo_promedio", "satisfaccion_usuario"
        ]
    ]

    tabla = tabla.replace(
        [np.inf, -np.inf], np.nan
    ).fillna("")

    return (
        kpis,
        fig_evolucion,
        fig_heatmap,
        fig_zonas,
        fig_zonas_treemap,
        fig_donut,
        fig_transporte,
        fig_box,
        fig_em_trans,
        fig_em_zona,
        fig_em_tiempo,
        fig_inc_tiempo,
        fig_inc_hora,
        insights,
        tabla.to_dict("records")
    )

# ============================================================
# 9. CSS
# ============================================================

CSS = r"""
body {
    margin: 0;
    background: #081018;
    color: #E8EEF5;
    font-family: Arial, Helvetica, sans-serif;
}

.dashboard {
    min-height: 100vh;
    background:
        radial-gradient(
            circle at 85% 0%,
            rgba(37, 99, 235, 0.12),
            transparent 30%
        ),
        radial-gradient(
            circle at 0% 25%,
            rgba(14, 165, 233, 0.07),
            transparent 25%
        ),
        #081018;
    padding: 28px 38px 45px 38px;
}

.dashboard-header {
    display: flex;
    justify-content: space-between;
    align-items: flex-end;
    border-bottom: 1px solid #263442;
    padding-bottom: 25px;
    margin-bottom: 24px;
}

.eyebrow {
    color: #5CC8FF;
    letter-spacing: 3px;
    font-size: 11px;
    font-weight: 700;
    margin-bottom: 8px;
}

h1 {
    margin: 0;
    font-size: 30px;
    letter-spacing: 0.5px;
    font-weight: 800;
}

.subtitle {
    margin: 9px 0 0 0;
    color: #98A9BA;
    font-size: 15px;
}

.header-period {
    text-align: right;
    padding: 12px 18px;
    border: 1px solid #263442;
    background: rgba(16, 24, 32, 0.78);
    border-radius: 12px;
}

.period-label {
    color: #7F92A4;
    font-size: 10px;
    letter-spacing: 1.5px;
    font-weight: 700;
}

.period-value {
    color: #FFFFFF;
    font-size: 14px;
    margin-top: 5px;
    font-weight: 700;
}

.period-records {
    color: #5CC8FF;
    font-size: 11px;
    margin-top: 5px;
}

.section-label {
    font-size: 12px;
    letter-spacing: 2px;
    color: #7890A5;
    font-weight: 800;
    margin: 28px 0 11px 3px;
}

.panel,
.half-panel {
    background: rgba(14, 23, 32, 0.88);
    border: 1px solid #263442;
    border-radius: 14px;
    box-shadow: 0 12px 35px rgba(0,0,0,0.18);
}

.graph-panel {
    padding: 10px 12px 4px 12px;
}

.filters-panel {
    padding: 18px;
}

.filters-grid {
    display: grid;
    grid-template-columns: 1.25fr repeat(5, 1fr);
    gap: 12px;
}

.filter-item label {
    display: block;
    color: #8EA1B3;
    font-size: 11px;
    margin-bottom: 6px;
    font-weight: 700;
}

.dash-dropdown .Select-control,
.Select-control {
    background-color: #111C26 !important;
    border: 1px solid #314353 !important;
    color: #E8EEF5 !important;
    border-radius: 8px !important;
}

.Select-menu-outer {
    background-color: #111C26 !important;
    color: #E8EEF5 !important;
    z-index: 9999 !important;
}

.Select-option {
    background-color: #111C26 !important;
    color: #E8EEF5 !important;
}

.Select-option:hover {
    background-color: #203342 !important;
}

.Select-value-label,
.Select--multi .Select-value-label {
    color: #E8EEF5 !important;
}

.Select-placeholder {
    color: #6E8295 !important;
}

.DateInput,
.DateInput_input {
    background-color: #111C26 !important;
    color: #E8EEF5 !important;
}

.DateRangePickerInput {
    background-color: #111C26 !important;
    border: 1px solid #314353 !important;
    border-radius: 8px;
}

.kpi-grid {
    display: grid;
    grid-template-columns: repeat(7, 1fr);
    gap: 11px;
}

.kpi-card {
    background:
        linear-gradient(
            145deg,
            rgba(20,34,47,0.98),
            rgba(12,21,30,0.98)
        );
    border: 1px solid #293B4B;
    border-radius: 14px;
    padding: 17px 15px;
    min-height: 105px;
    position: relative;
    overflow: hidden;
}

.kpi-card:before {
    content: "";
    position: absolute;
    left: 0;
    top: 0;
    width: 4px;
    height: 100%;
    background: #36BDF8;
}

.kpi-title {
    color: #7890A5;
    font-size: 10px;
    letter-spacing: 1.2px;
    font-weight: 800;
}

.kpi-value {
    color: #F6FAFD;
    font-size: 22px;
    font-weight: 800;
    margin-top: 10px;
    white-space: nowrap;
}

.kpi-subtitle {
    color: #71879A;
    font-size: 10px;
    margin-top: 6px;
}

.two-columns {
    display: grid;
    grid-template-columns: 1fr 1fr;
    gap: 14px;
}

.half-panel {
    padding: 10px 12px 4px 12px;
}

.chart-description {
    color: #7F92A4;
    font-size: 12px;
    margin: 3px 10px 0 10px;
}

.insights-grid {
    display: grid;
    grid-template-columns: repeat(3, 1fr);
    gap: 12px;
}

.insight-item {
    display: flex;
    gap: 13px;
    background: #101A24;
    border: 1px solid #273847;
    border-radius: 12px;
    padding: 16px;
    min-height: 82px;
}

.insight-number {
    color: #36BDF8;
    font-size: 12px;
    font-weight: 900;
    padding-top: 2px;
}

.insight-title {
    color: #EAF3F9;
    font-weight: 800;
    font-size: 13px;
    margin-bottom: 6px;
}

.insight-text {
    color: #899CAD;
    font-size: 12px;
    line-height: 1.5;
}

.insight-empty {
    padding: 20px;
    color: #8A9CAE;
}

.table-panel {
    padding: 14px;
}

.footer {
    text-align: center;
    color: #536678;
    font-size: 10px;
    letter-spacing: 0.7px;
    margin-top: 35px;
}

@media (max-width: 1200px) {
    .filters-grid {
        grid-template-columns: repeat(3, 1fr);
    }

    .kpi-grid {
        grid-template-columns: repeat(4, 1fr);
    }

    .insights-grid {
        grid-template-columns: repeat(2, 1fr);
    }
}

@media (max-width: 800px) {
    .dashboard {
        padding: 18px;
    }

    .dashboard-header {
        flex-direction: column;
        align-items: flex-start;
        gap: 18px;
    }

    .header-period {
        text-align: left;
    }

    .filters-grid,
    .two-columns,
    .insights-grid,
    .kpi-grid {
        grid-template-columns: 1fr;
    }
}
"""

app.index_string = f"""
<!DOCTYPE html>
<html>
<head>
    {{%metas%}}
    <title>Ciudad Nova | Movilidad Inteligente</title>
    {{%favicon%}}
    {{%css%}}
    <style>{CSS}</style>
</head>
<body>
    {{%app_entry%}}
    <footer>
        {{%config%}}
        {{%scripts%}}
        {{%renderer%}}
    </footer>
</body>
</html>
"""

# ============================================================
# ============================================================
# 10. EJECUCIÓN EN GOOGLE COLAB — VENTANA EXTERNA
# ============================================================
# Este bloque inicia Dash en segundo plano y abre el dashboard
# en una ventana/pestaña externa mediante Google Colab.

import threading
import time
from google.colab import output

PUERTO = 8050

def ejecutar_dashboard():
    app.run(
        debug=False,
        use_reloader=False,
        host="0.0.0.0",
        port=PUERTO
    )

print("=" * 70)
print("INICIANDO DASHBOARD — CIUDAD NOVA")
print("=" * 70)
print(f"Registros generados: {len(df):,}")
print(f"Periodo: {FECHA_INICIO:%d/%m/%Y} a {FECHA_FIN:%d/%m/%Y}")
print("Iniciando servidor Dash...")
print("=" * 70)

# Iniciar el servidor una sola vez.
hilo_dashboard = threading.Thread(
    target=ejecutar_dashboard,
    daemon=True
)
hilo_dashboard.start()

# Dar tiempo al servidor para arrancar.
time.sleep(5)

print("Servidor iniciado en el puerto 8050.")
print("Abriendo el dashboard en una ventana/pestaña externa...")
print()
print("Si el navegador bloquea la ventana emergente, permite las ventanas")
print("emergentes para Google Colab y vuelve a ejecutar SOLO esta sección.")

# Abrir Dash fuera del área de salida de Colab.
output.serve_kernel_port_as_window(
    PUERTO
)


Dataset creado correctamente: 100,000 registros.
Período: 01/09/2025 — 31/08/2026
Columnas: 31
INICIANDO DASHBOARD — CIUDAD NOVA
Registros generados: 100,000
Periodo: 01/09/2025 a 31/08/2026
Iniciando servidor Dash...


<IPython.core.display.Javascript object>

Servidor iniciado en el puerto 8050.
Abriendo el dashboard en una ventana/pestaña externa...

Si el navegador bloquea la ventana emergente, permite las ventanas
emergentes para Google Colab y vuelve a ejecutar SOLO esta sección.
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>